### 1. Configuração Inicial e Carregamento de Dados
**Objetivo:** Preparar o ambiente de análise, importando as bibliotecas necessárias e carregando o conjunto de dados inicial.
Fonte de Dados: Banco de dados relacional.

In [1]:
# --- Importação de Bibliotecas Padrão e de Terceiros ---
import os
import re
import ast
import sys
import unidecode
import pandas as pd

# --- Configuração do Caminho do Projeto para Importações Locais ---
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# --- Importação de Módulos e Funções Customizadas do Projeto ---
from src.utils.data_cleaning import normalize_text_columns
from src.utils.number_utils import br_to_float
from src.services.db_manager import fetch_data_from_db
from src.config import const

In [25]:
# Buscar os dados no banco de dados
df = fetch_data_from_db(const.consulta_sql)

In [3]:
df.head()

,id,orgaoDestinatario,nomeFornecedor,cnpjFornecedor,municipioFornecedor,valorNotaFiscal,itensNotaFiscal,tipoEventoMaisRecente,dataEmissao,eventosNotaFiscal,chaveNotaFiscal
0,1949,Ministério da Saúde - Unidades com vínculo direto,BRISTOL-MYERS SQUIBB FARMACEUTICA LTDA,56.998.982/0001-07,SÃO PAULO,"4.237,13","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,16/11/2021,[],35211156998982000107550010000317171081856667
1,1950,Ministério da Saúde - Unidades com vínculo direto,RCA PRODUTOS E SERVICOS LTDA,69.207.850/0001-61,SANTA BARBARA D'OESTE,"16.004,55","[{'numeroProduto': '45', 'descricaoProdutoServ...",Autorização de Uso,01/11/2021,[],35211169207850000161550010000008501783475076
2,1967,Ministério da Saúde - Unidades com vínculo direto,ATHOS RIO PRODUTOS MEDICOS HOSPITALARES LTDA,31.912.939/0001-56,MESQUITA,"3.400,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,05/11/2021,[],33211131912939000156550010000008061429011828
3,1981,Ministério da Saúde - Unidades com vínculo direto,CLARO S.A,40.432.544/0644-63,MANAUS,"2.810,32","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,08/11/2021,[],13211140432544064463554090000040661710617220
4,1984,Ministério da Saúde - Unidades com vínculo direto,ANJOMEDI DISTRIBUIDORA DE MEDICAMENTOS LTDA,31.151.224/0001-28,ERECHIM,"450,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,01/11/2021,[],43211131151224000128550010000064451579723624


### 2. Tratamento e Junção de Dados Demográficos

**Objetivo:** Adicionar o índice populacional de cada município ao DataFrame de trabalho.
**Fonte de Dados:** Arquivo com a população municipal, extraído do IBGE.

**Passos:**
1.  **Carregamento:** Leitura do arquivo de dados populacionais.
2.  **Tratamento:** Seleção das colunas de interesse (`código_ibge`, `populacao`,`UF`) e padronização dos tipos de dados.
3.  **Junção (`Merge`):** União do DataFrame de população com o DataFrame principal, usando o código IBGE como chave de ligação.

In [4]:
# Analisando os valores únicos na coluna 'municipioFornecedor'
df[df["municipioFornecedor"]=="CONTATO@FENICEMEDICAL.COM.BR"]

,id,orgaoDestinatario,nomeFornecedor,cnpjFornecedor,municipioFornecedor,valorNotaFiscal,itensNotaFiscal,tipoEventoMaisRecente,dataEmissao,eventosNotaFiscal,chaveNotaFiscal
123593,188402291,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"7,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,23/03/2023,[],33230339800235000101550010000000401088076081
134770,194001045,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"269,08","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,09/12/2022,[],33221239800235000101550010000000141088076089
165022,208802074,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"3.384,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,01/03/2023,[],33230339800235000101550010000000361088076089
165252,209001079,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"115,32","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,12/12/2022,[],33221239800235000101550010000000151088076086
176215,214402232,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"1.720,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Carta de correção,14/03/2023,"[{'dataEvento': '14/03/2023 12:59:29', 'tipoEv...",33230339800235000101550010000000391088076080
...,...,...,...,...,...,...,...,...,...,...,...
394647,247311829,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"5.400,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,26/02/2025,[],33250239800235000101550010000007771114011393
394842,247314410,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"3.000,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,27/02/2025,[],33250239800235000101550010000007791870087917
402758,247410204,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"3.200,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,20/03/2025,[],33250339800235000101550010000008121675067635
408313,247493901,Ministério da Saúde - Unidades com vínculo direto,FENICE COMERCIO E SERVICOS LTDA,39.800.235/0001-01,CONTATO@FENICEMEDICAL.COM.BR,"4.000,00","[{'numeroProduto': '1', 'descricaoProdutoServi...",Autorização de Uso,11/04/2025,[],33250439800235000101550010000008671960096852


2.1 **Carregamento:**

In [5]:
# Carregar os dados demográficos dos municípios
mun = pd.read_csv(
    r"../data/raw/ibge_municipios.csv",
    sep=";",
    encoding="utf-8"
)

2.2 **Tratamento**

In [6]:
# Normalização: Deixa os nomes em Letra maiuscula e remove a acentuação
mun = normalize_text_columns(mun)
mun["COD. IBGE"] = mun["COD. UF"].astype(str).str.zfill(2) + mun["COD. MUNIC"].astype(str).str.zfill(5)
df['municipioFornecedor'] = df['municipioFornecedor'].apply(lambda x: unidecode.unidecode(x.upper()))

In [7]:
# Verificando quais municípios do DataFrame original não existem no DataFrame de municípios

mun['municipio']=mun['NOME DO MUNICÍPIO']

mun_set = set(mun['municipio'])
df2 = df.copy()
df2['existe_em_mun'] = df['municipioFornecedor'].apply(lambda x: x in mun_set)

nao_encontrados = df[~df2['existe_em_mun']]['municipioFornecedor'].unique()
print("Municípios não encontrados:")
print(nao_encontrados)

Municípios não encontrados:
['EMBU' 'SANTA ISABEL DO PARA' 'APARECIDA GOIANIA' 'ANAPOLIS-GO'
 'JI PARANA' 'POXOREO' 'AGUAS LINDAS DE GO' 'S. J. RIO PRETO' 'SO LUIS'
 'CAJAMAR - SP' 'BRASILIA (DF)' 'SAO  PAULO' 'SAO BERNADO DO CAMPO'
 '5300108' 'BELEM - PA' 'S PAULO' 'S LUIS  MONTES BELOS'
 'J. DOS GUARARAPES' 'FORTALEZA (CE)' 'AP GOIANIA' 'SERRA (ES)'
 'COUTO DE MAGALHAES' 'CUIABA - MT' 'SAO BERNARDO CAMPO'
 'NUCLEO BANDEIRANTE' 'GUAJARA MIRIM' 'JOAO PESSOA - PB'
 'SAO JORGE D OESTE' 'ARACAJU,' 'JABOTAO DOS GARARAPES' 'FOTALEZA'
 'SANTA RITA SAPUCAI' 'SANTANA DO PARNAIBA' 'RECIFE - PE'
 'ITAIPAVA - ITAJAI' 'CONTATO@FENICEMEDICAL.COM.BR' 'ANANINDEUA.'
 'BRASILA-DF' 'MOGI DAS CRUZES - SAO PAULO'
 'ESTANCIA TURISTICA DE OLIMPIA' 'BELEM (PA)' 'ALTA FLORESTA D OESTE'
 'ELDORADO DOS CARAJAS' 'GOV VALADARES' 'JOINVILE' 'REDENAEAO' 'BRASILI'
 'IMPETRATRIZ' 'PENHA                        ' 'S.J. RIO PRETO'
 'MOJI MIRIM' 'MOGI-MIRIM' 'TERESINA PI' 'SAO PAULO - SP'
 'ESPIGAO D OESTE' 'TERESINA (PI

In [8]:
# Correções manuais para alguns municípios conhecidos

ajustes = {
    'EMBU': 'EMBU DAS ARTES',
    'SANTA ISABEL DO PARA': 'SANTA ISABEL DO PARÁ',
    'APARECIDA GOIANIA': 'APARECIDA DE GOIÂNIA',
    'ANAPOLIS-GO': 'ANÁPOLIS',
    'JI PARANA': 'JI-PARANÁ',
    'POXOREO': 'POXORÉU',
    'AGUAS LINDAS DE GO': 'ÁGUAS LINDAS DE GOIÁS',
    'S. J. RIO PRETO': 'SÃO JOSÉ DO RIO PRETO',
    'SO LUIS': 'SÃO LUÍS',
    'CAJAMAR - SP': 'CAJAMAR',
    'BRASILIA (DF)': 'BRASÍLIA',
    'SAO  PAULO': 'SÃO PAULO',
    'SAO BERNADO DO CAMPO': 'SÃO BERNARDO DO CAMPO',
    '5300108': 'BRASÍLIA',
    'BELEM - PA': 'BELÉM',
    'S PAULO': 'SÃO PAULO',
    'S LUIS  MONTES BELOS': 'SÃO LUÍS DE MONTES BELOS',
    'J. DOS GUARARAPES': 'JABOATÃO DOS GUARARAPES',
    'FORTALEZA (CE)': 'FORTALEZA',
    'AP GOIANIA': 'APARECIDA DE GOIÂNIA',
    'SERRA (ES)': 'SERRA',
    'COUTO DE MAGALHAES': 'COUTO DE MAGALHÃES',
    'CUIABA - MT': 'CUIABÁ',
    'SAO BERNARDO CAMPO': 'SÃO BERNARDO DO CAMPO',
    'NUCLEO BANDEIRANTE': 'BRASÍLIA',  # região administrativa
    'GUAJARA MIRIM': 'GUAJARÁ-MIRIM',
    'JOAO PESSOA - PB': 'JOÃO PESSOA',
    'SAO JORGE D OESTE': 'SÃO JORGE D’OESTE',
    'ARACAJU,': 'ARACAJU',
    'JABOTAO DOS GARARAPES': 'JABOATÃO DOS GUARARAPES',
    'FOTALEZA': 'FORTALEZA',
    'SANTA RITA SAPUCAI': 'SANTA RITA DO SAPUCAÍ',
    'SANTANA DO PARNAIBA': 'SANTANA DE PARNAÍBA',
    'RECIFE - PE': 'RECIFE',
    'ITAIPAVA - ITAJAI': 'ITAJAI',
    'CONTATO@FENICEMEDICAL.COM.BR': 'RIO DE JANEIRO',
    'ANANINDEUA.': 'ANANINDEUA',
    'BRASILA-DF': 'BRASÍLIA',
    'MOGI DAS CRUZES - SAO PAULO': 'MOGI DAS CRUZES',
    'ESTANCIA TURISTICA DE OLIMPIA': 'OLÍMPIA',
    'BELEM (PA)': 'BELÉM',
    'ALTA FLORESTA D OESTE': 'ALTA FLORESTA D’OESTE',
    'ELDORADO DOS CARAJAS': 'ELDORADO DOS CARAJÁS',
    'GOV VALADARES': 'GOVERNADOR VALADARES',
    'JOINVILE': 'JOINVILLE',
    'REDENAEAO': 'REDENÇÃO',
    'BRASILI': 'BRASÍLIA',
    'IMPETRATRIZ': 'IMPERATRIZ',
    'PENHA                        ': 'PENHA',
    'S.J. RIO PRETO': 'SÃO JOSÉ DO RIO PRETO',
    'MOJI MIRIM': 'MOGI MIRIM',
    'MOGI-MIRIM': 'MOGI MIRIM',
    'TERESINA PI': 'TERESINA',
    'SAO PAULO - SP': 'SÃO PAULO',
    'ESPIGAO D OESTE': 'ESPIGÃO D’OESTE',
    'TERESINA (PI)': 'TERESINA',
    'BAL. CAMBORIU': 'BALNEÁRIO CAMBORIÚ',
    'SALVADOR (BA)': 'SALVADOR',
    'MOJI-MIRIM': 'MOGI MIRIM',
    'BALNEARIO PI&CCEDIL;ARRAS': 'BALNEÁRIO PIÇARRAS',
    'NOSSA SENHORA DO SOC,': 'NOSSA SENHORA DO SOCORRO',
    'ALTA FLORESTA DOESTE': 'ALTA FLORESTA D’OESTE',
    'PAU DARCO': 'PAU D’ARCO',
    'SANTO ANTONIO DE LEVERGER': 'SANTO ANTÔNIO DO LEVERGER'
}

df['municipioFornecedor'] = df['municipioFornecedor'].replace(ajustes)


2.3 **Junção (`Merge`)**

In [9]:
df = df.merge(
    mun[['municipio', 'UF', 'POPULAÇÃO ESTIMADA', 'COD. IBGE']],
    left_on='municipioFornecedor',
    right_on='municipio',
    how='left'
)

## 3. Engenharia de Features e Ajustes Finais
**Objetivo:** Limpar, transformar e preparar as colunas do DataFrame principal para a fase de análise, garantindo a consistência e o formato correto dos dados.

**Passos:**
1.  **Tratamento das features:** Realizando tratamento inicial das colunas para melhor vizualização e melhor exploração dos dados.
2.  **Criando uma tabebela auxiliar:** Criar um DataFrame auxiliar (`df_itens`) onde cada linha representa um único item de uma nota fiscal, permitindo análises em nível de produto.
3.  **Separação das Features da coluna Auxiliar** filtramos as colunas mais importantes para a tabela auxiliar

3.1 **Tratamento das features:**

In [10]:
# Renomear colunas para nomes mais amigáveis
df = df.rename(columns={
    'id': 'ID',
    'orgaoDestinatario': 'ORGAO',
    'nomeFornecedor': 'FORNECEDOR',
    'cnpjFornecedor': 'CNPJ',
    'municipioFornecedor': 'MUNICIPIO',
    "valorNotaFiscal" : 'VALOR_NF',
    'itensNotaFiscal': 'ITENS',
    'tipoEventoMaisRecente': 'TIPO_EVENTO',
    'dataEmissao': 'DATA',
    'eventosNotaFiscal': 'EVENTOS',
    'chaveNotaFiscal': 'CHAVE_NF',
    'municipio': 'MUNICIPIO_MUN',
    'UF': 'UF',
    'POPULAÇÃO ESTIMADA': 'POPULACAO',
    'COD. IBGE': 'COD_IBGE'
})

In [11]:
# Remove tudo que não é número
df['CNPJ'] = df['CNPJ'].apply(lambda x: re.sub(r'\D', '', str(x)))

In [12]:
# Converter VALOR_NF de string para float
df['VALOR_NF'] = df['VALOR_NF'].str.replace('.', '', regex=False)  # remove separador de milhar
df['VALOR_NF'] = df['VALOR_NF'].str.replace(',', '.', regex=False)  # converte vírgula decimal para ponto
df['VALOR_NF'] = pd.to_numeric(df['VALOR_NF'], errors='coerce')


In [13]:
# Garantir que POPULACAO seja numérico
df['POPULACAO'] = pd.to_numeric(df['POPULACAO'], errors='coerce')

In [14]:
# Remover linhas inválidas
df = df.dropna(subset=['VALOR_NF', 'POPULACAO'])

In [15]:
# Converter DATA para o formato datetime
df['DATA'] = pd.to_datetime(df['DATA'], dayfirst=True)

# Criar coluna Ano-Mês
df['ANO_MES'] = df['DATA'].dt.to_period('M')

3.2 **Criando uma tabebela auxiliar:**

In [16]:
import ast
import pandas as pd

def safe_eval(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x == "" or x.lower() in ["nan", "none", "null"]:
            return []
        try:
            return ast.literal_eval(x)
        except Exception:
            return []
    return []


In [17]:
df['ITENS'] = df['ITENS'].apply(safe_eval)

In [18]:

# Explodir a lista em várias linhas (cada item vira uma linha)
df_itens = df.explode('ITENS')

# Transformar o dicionário de cada item em colunas
df_itens = pd.concat([df_itens.drop(columns=['ITENS']),
                      df_itens['ITENS'].apply(pd.Series)], axis=1)

# Selecionar apenas as colunas que você quer manter + CHAVE_NF
colunas_desejadas = ['CHAVE_NF', 'descricaoProdutoServico', 'codigoNcmSh', 
                     'ncmSh', 'cfop', 'quantidade', 'unidade', 'valorUnitario', 'valor']

df_itens_aux = df_itens[colunas_desejadas]

df_itens['quantidade'] = df_itens['quantidade'].apply(br_to_float)
df_itens['valorUnitario'] = df_itens['valorUnitario'].apply(br_to_float)
df_itens['valor'] = df_itens['valor'].apply(br_to_float)



3.3 **Separação das Features da coluna Auxiliar**

In [19]:
colunas_desejadas = ['descricaoProdutoServico', 'CHAVE_NF', 'codigoNcmSh', 'ncmSh', 
                     'cfop', 'quantidade', 'unidade', 'valorUnitario', 'valor']

df_itens = df_itens[colunas_desejadas]

In [20]:
df_itens.sample(2)

,descricaoProdutoServico,CHAVE_NF,codigoNcmSh,ncmSh,cfop,quantidade,unidade,valorUnitario,valor
364708,EQUIPO DE BOMBA INFUSORA P/NUTRICAO ENTERAL,33240810839887000160550000000117861043310331,90183929,"Outras sondas, catéteres e cânulas",5106,1000.0,PC,42.60,42600.00
317414,MA60AC.280 ACRYSOF MP DOBRAVEL 6.0 OPTIC,52240432929819000477550010005976361917088879,90213920,Lentes intraoculares,6949,1.0,UN,196.41,196.41


In [21]:
df.dtypes

ID                       object
ORGAO                    object
FORNECEDOR               object
CNPJ                     object
MUNICIPIO                object
VALOR_NF                float64
ITENS                    object
TIPO_EVENTO              object
DATA             datetime64[ns]
EVENTOS                  object
CHAVE_NF                 object
MUNICIPIO_MUN            object
UF                       object
POPULACAO               float64
COD_IBGE                 object
ANO_MES               period[M]
dtype: object

## 4. Exportação dos Dataframes para o formato .csv
**Informativo:** Agora que relizado o pré-processamento dos dados, os mesmo serão exportados para o formato .csv para fins da Análise Exploratória dos Dados

In [ ]:
# Definir diretório de saída (pasta processed dentro de data)
output_dir = os.path.join(project_root, "data", "processed")


# Exportar df
df.to_csv(
    os.path.join(output_dir, "notas_fiscais.csv"),
    index=False,
    sep=";", 
    encoding="utf-8"
)

# Exportar df_itens detalhado
df_itens.to_csv(
    os.path.join(output_dir, "itens_notas_fiscais.csv"),
    index=False,
    sep=";", 
    encoding="utf-8"
)